# Extract stops for users living in region of interest (roi) during period of interest (poi).

<span style='color:red'>Note, used 'provider_id = 190199' here, because very slow query</span>

In [ ]:
year = '2019'
month = '04'

roi = 'NYC'

min_stops = 30

process_lag = 3 # WORKS WELL FOR PROVIDER_ID 190199, BUT SHOULD PROB BE CHANGED IF USING OTHER PROVIDERS
date_format = "%Y%m%d"

In [ ]:
# SQL engine
from trino.dbapi import connect 
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import time
import networkx as nx
import calendar
from datetime import datetime, timedelta

class TrinoEngine():
    def __init__(self):
        conn = connect(
            host="localhost",
            port=9090,
            catalog="data_provider" # removed data provider name to preserve anonymity
        )
        self.cur = conn.cursor()
        self.engine = create_engine("trino://localhost:9090/data_provider/") removed data provider name to preserve anonymity
    
    def execute_statement(self, query:str) -> list:
        """
        Create and drop statements.
        """
        self.cur.execute(query)
        return self.cur.fetchall()
    
    def read_sql(self, query:str) -> pd.DataFrame: 
        """
        Select and insert into operations.
        """
        return pd.read_sql(query, self.engine)

sql_engine = TrinoEngine()

In [ ]:
dpoints_homelocs = pd.read_csv(f'dpoints_homelocs/{roi}_{year}_{month}.csv')

In [ ]:
dpoints_homelocs

In [ ]:
schema_name = {'cda': 'data_provider.paas_cda_pe_v3'}
stop_table = f"{schema_name['cda']}.stop_uplevelled"

In [ ]:
users = list(dpoints_homelocs.user_id.map(lambda x: str(x)))
users_str = ','.join(str(id) for id in users)

del dpoints_homelocs

print(f'users:   {len(users)}\nstring: {len(users_str)}')

In [ ]:
# adds process_lag days to given date
def lag_date(date, process_lag, date_format):
    return datetime.strftime(datetime.strptime(date, date_format) + timedelta(days=process_lag), date_format)

start_date = f'{year}{month}01'
end_date = lag_date(f'{year}{month}{calendar.monthrange(int(year), int(month))[1]}', process_lag, date_format)

print(f'start_date: {start_date}\nend_date:   {end_date}')

In [ ]:
%%time
output = []
grp_size = int(len(users) / 6)
for i in range(6):
    part_users = users[i * grp_size : (i + 1) * grp_size]
    part_users_str = ','.join(str(id) for id in part_users)
    output.append(sql_engine.read_sql(
        f"""
        select 
            user_id,
            block_group_id,
            stop_zoned_datetime,
            dwell_time_minutes
        from {stop_table}
        where 
            country_code = 'US'
            and user_id in ({part_users_str})
            and processing_date between {start_date} and {end_date}
            and substr(stop_zoned_datetime,1,7) = '{year}-{month}'
            and provider_id = '190199'
        """
    ))
    display(output[i])

In [ ]:
stops = pd.concat(output)
display(stops)

stops.to_csv(f'nw_extracts/Stops_TEMP_{roi}_{year}_{month}.csv', index=False) # save before you loose everything by crash

In [ ]:
# # if loading in directly
# stops = pd.read_csv(f'nw_extracts/Stops_TEMP_{roi}_{year}_{month}.csv')

In [ ]:
# only keep user_id's with no NaN values and at least X occurences
stops = stops[~stops.isnull().any(axis=1)]
total_users = len(stops.user_id.unique())
print(f'Users totally extracted: {total_users}')

stops = stops.groupby('user_id').filter(lambda x: len(x) >= min_stops).reset_index(drop=True)
print(f'Outliers removed (<{min_stops} stops): {total_users - len(stops.user_id.unique())}')
print(f'Users left: {len(stops.user_id.unique())}')
display(stops)

In [ ]:
stops.to_csv(f'nw_extracts/Stops_{roi}_{year}_{month}.csv', index=False)